In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import joblib



In [2]:
import os
import pandas as pd

file_path = r"C:\Users\bohar\Desktop\archive\training.1600000.processed.noemoticon.csv"

if not os.path.exists(file_path):
    print("❌ File not found. Please check your path:")
    print(file_path)
else:
    print("✅ File found! Loading now...")
    try:
        df = pd.read_csv(file_path, encoding='latin-1', on_bad_lines='skip')
        print("✅ File loaded successfully!")
        print("Number of rows:", len(df))
    except Exception as e:
        print("⚠️ Error while loading file:", e)




✅ File found! Loading now...
✅ File loaded successfully!
Number of rows: 1599999


In [3]:
df.columns = ["target", "ids", "date", "flag", "user", "text"]

df = df[["text", "target"]]

df["label"] = df["target"].map({0: 0, 4: 1})

df = df.drop("target", axis=1)

df = df.dropna(subset=["text"])
df = df[df["text"].str.strip() != ""]

print("✅ Data cleaned successfully!")
print(df.head())


✅ Data cleaned successfully!
                                                text  label
0  is upset that he can't update his Facebook by ...      0
1  @Kenichan I dived many times for the ball. Man...      0
2    my whole body feels itchy and like its on fire       0
3  @nationwideclass no, it's not behaving at all....      0
4                      @Kwesidei not the whole crew       0


In [4]:
df = df.sample(50000, random_state=42)

print("✅ Sampled 50,000 rows for faster training.")
print(df["label"].value_counts())


✅ Sampled 50,000 rows for faster training.
label
1    25014
0    24986
Name: count, dtype: int64


In [5]:
from sklearn.model_selection import train_test_split

X = df["text"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("✅ Training samples:", len(X_train))
print("✅ Testing samples:", len(X_test))


✅ Training samples: 40000
✅ Testing samples: 10000


In [6]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

pipe = Pipeline([
    ('vectorizer', CountVectorizer(stop_words='english')),
    ('classifier', LogisticRegression(max_iter=1000))
])

pipe.fit(X_train, y_train)

print("✅ Model trained successfully!")


✅ Model trained successfully!


In [13]:
y_pred = pipe.predict(X_test)

accuracy = accuracy_score(y_test, y_pred) * 100  
rounded_accuracy = round(accuracy, 2)          

print(f"✅ Model Accuracy: {rounded_accuracy}%")
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))


✅ Model Accuracy: 74.46%

Confusion Matrix:
 [[3624 1353]
 [1201 3822]]

Classification Report:
               precision    recall  f1-score   support

           0       0.75      0.73      0.74      4977
           1       0.74      0.76      0.75      5023

    accuracy                           0.74     10000
   macro avg       0.74      0.74      0.74     10000
weighted avg       0.74      0.74      0.74     10000



In [11]:
samples = [
    "I love your dress",
    "I hate this service",
    "The food was okay, not bad",
    "This phone is terrible",
    "Amazing experience with the app!"
]

predictions = pipe.predict(samples)

for text, label in zip(samples, predictions):
    sentiment = "Positive 😄" if label == 1 else "Negative 😞"
    print(f"{text} → {sentiment}")



I love your dress → Positive 😄
I hate this service → Negative 😞
The food was okay, not bad → Negative 😞
This phone is terrible → Negative 😞
Amazing experience with the app! → Positive 😄


In [12]:
import joblib

joblib.dump(pipe, "sentiment_model.joblib")
print("✅ Model saved successfully as sentiment_model.joblib")



✅ Model saved successfully as sentiment_model.joblib
